# EntityLink AI — Data Profiling & Candidate Recall Benchmarks
**Author:** Adi (Data Engineering & Blocking Owner)

This notebook demonstrates:
1. Loading TSV datasets and schema profiling
2. Text normalization (raw vs. normalized, compact, tokens)
3. Multi-strategy candidate generation
4. Candidate recall and distribution metrics

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))

from entitylink.data.loader import load_records_dict, load_ground_truth
from entitylink.preprocessing.normalizer import build_normalized_record
from entitylink.blocking.candidate_generator import generate_candidates
from entitylink.evaluation.metrics import evaluate_blocking_performance

## 1. Inspect Text Normalization

In [ ]:
sample_name = 'Acme Corporation, LLC'
sample_addr = '123 Main Street Suite 100'
rec = build_normalized_record('S1-001', sample_name, sample_addr, 'USA')

print(f'Raw Name:        {rec.business_name_raw}')
print(f'Normalized Name: {rec.business_name_normalized}')
print(f'Compact Name:    {rec.business_name_compact}')
print(f'Tokens Name:     {rec.business_name_tokens}')
print(f'Normalized Addr: {rec.business_address_normalized}')
print(f'Country:         {rec.country_normalized}')

## 2. Generate Candidates & Compute Recall

In [ ]:
fixture_dir = '../tests/fixtures/train'
if os.path.exists(fixture_dir):
    s1 = load_records_dict(os.path.join(fixture_dir, 'train_source1.tsv'))
    s2 = load_records_dict(os.path.join(fixture_dir, 'train_source2.tsv'))
    s3 = load_records_dict(os.path.join(fixture_dir, 'train_source3.tsv'))
    gt = load_ground_truth(os.path.join(fixture_dir, 'train_ground_truth.tsv'))
    
    cands = generate_candidates(s1, s2, s3)
    report = evaluate_blocking_performance(gt, cands, total_target_records=len(s2)+len(s3))
    
    print(f'Candidate Recall:  {report.candidate_recall * 100:.2f}%')
    print(f'Avg Candidates/S1: {report.avg_candidates_per_entity:.2f}')
    print(f'Reduction Ratio:   {report.reduction_ratio * 100:.4f}%')